In [16]:
import pandas as pd

df = pd.read_csv("../data/raw/House_Price_Data.csv")

In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14119 entries, 0 to 14118
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   bhk           14119 non-null  int64
 1   propertytype  14119 non-null  str  
 2   location      14119 non-null  str  
 3   sqft          14119 non-null  int64
 4   pricepersqft  14119 non-null  int64
 5   totalprice    14119 non-null  int64
dtypes: int64(4), str(2)
memory usage: 662.0 KB


In [18]:
from sklearn.model_selection import GroupShuffleSplit

feature_columns = ["bhk", "propertytype", "location", "sqft"]

groups = df[feature_columns].astype(str).agg("|".join, axis=1)

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(df, groups=groups)
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (11302, 6)
Test shape: (2817, 6)


In [19]:
train_keys = set(
    train_df[feature_columns].astype(str).agg("|".join, axis=1)
)

test_keys = set(
    test_df[feature_columns].astype(str).agg("|".join, axis=1)
)

overlapping_keys = train_keys.intersection(test_keys)

print("Unique feature groups in train:", len(train_keys))
print("Unique feature groups in test:", len(test_keys))
print("Overlapping feature groups:", len(overlapping_keys))

Unique feature groups in train: 7536
Unique feature groups in test: 1884
Overlapping feature groups: 0


In [20]:
print("Train target statistics:")
print(train_df["totalprice"].describe())

print("\nTest target statistics:")
print(test_df["totalprice"].describe())

Train target statistics:
count    1.130200e+04
mean     1.282791e+07
std      2.043398e+07
min      1.100000e+05
25%      4.500000e+06
50%      7.500000e+06
75%      1.450000e+07
max      7.800000e+08
Name: totalprice, dtype: float64

Test target statistics:
count    2.817000e+03
mean     1.252259e+07
std      2.367405e+07
min      1.500000e+05
25%      4.500000e+06
50%      7.300000e+06
75%      1.300000e+07
max      8.686000e+08
Name: totalprice, dtype: float64


In [21]:
import numpy as np

price_check = (
    df["totalprice"] / df["sqft"]
)

difference = (
    df["pricepersqft"] - price_check
).abs()

print("Mean absolute difference:", difference.mean())
print("Median absolute difference:", difference.median())
print("Maximum absolute difference:", difference.max())
print("Exact matches:", (difference == 0).sum())
print("Total rows:", len(df))

Mean absolute difference: 12217.536909301947
Median absolute difference: 1937.5263157894733
Maximum absolute difference: 19996000.0
Exact matches: 79
Total rows: 14119
